# Assignment 4 - Coding
### Andres Arias Medina
### Z23987588

### I used Claude to help me generate some markdowns with mathematical expressions

## 9.RMSProp
### Setup
Import numpy and define the objective function f(x, y) = 2x² + 3y² + 4x + 5y + 1.

In [1]:
import numpy as np  # numerical operations (sqrt)

def f(x, y):  # objective function to minimize
    return 2*x**2 + 3*y**2 + 4*x + 5*y + 1  # f(x, y) = 2x^2 + 3y^2 + 4x + 5y + 1

### 9. 1)-1 Derivative with respect to x

In [2]:
# Compute the derivative with respect to x
# f(x, y) = 2x^2 + 3y^2 + 4x + 5y + 1
def f_der_x(x):  # partial derivative of f with respect to x
    return 4*x + 4  # d/dx (2x^2 + 4x) = 4x + 4; the y terms are constant w.r.t. x

### 9. 1)-2 Derivative with respect to y

In [3]:
# Compute the derivative with respect to y
def f_der_y(y):
    return 6*y + 5  # d/dy (3y^2 + 5y) = 6y + 5; the x terms are constant w.r.t. y

### 9. 1)-3 RMSProp optimization loop

In every iteration, for each variable:
1. Compute the gradient at the current point with `f_der_x` / `f_der_y`.
2. Update the moving average of squared gradients:
   $G_t = \gamma G_{t-1} + (1-\gamma)\left(\nabla_w J(w_t)\right)^2$ (`gt_x`, `gt_y`).
3. Update the variable:
   $w_{t+1} = w_t - \frac{\eta}{\sqrt{G_t + \epsilon}}\nabla_w J(w_t)$.

In [4]:
# Implement RMSProp

# RMSProp parameters
# γ: gamma
# η: lrate
# ϵ: epsilon
gamma = 0.8  
lrate = 0.1
epsilon = 1e-4

# Accumulated gradient squares for x and y, respectively
# G_t: gt_x and gt_y
gt_x = 0  
gt_y = 0 

# Initial values of x and y
x, y = 10.0, -10.
no_iterations = 100

# RMSProp optimization loop
for i in range(no_iterations):
    # Compute gradients
    grad_x = f_der_x(x)
    grad_y = f_der_y(y)

    # Update accumulated gradient squares
    gt_x = gamma * gt_x + (1 - gamma) * grad_x**2
    gt_y = gamma * gt_y + (1 - gamma) * grad_y**2

    # Update parameters x and y using RMSProp update rule
    x -= lrate / np.sqrt(gt_x + epsilon) * grad_x  # RMSProp update: x = x − η/√(G_t + ϵ)·∂f/∂x
    y -= lrate / (np.sqrt(gt_y + epsilon)) * grad_y

    print(f"Iteration = {i + 1}: x = {x}, y = {y}, f(x, y) = {f(x, y)}")  # show progress at each iteration


Iteration = 1: x = 9.77639323112486, y = -9.776393220729918, f(x, y) = 469.11292925901955
Iteration = 2: x = 9.611258163085816, y = -9.61157075168111, f(x, y) = 453.2866227926206
Iteration = 3: x = 9.470198729449011, y = -9.470940777802968, f(x, y) = 439.9915766292498
Iteration = 4: x = 9.342471534021069, y = -9.343714081411665, f(x, y) = 428.12984296252586
Iteration = 5: x = 9.223166288309438, y = -9.224962523848532, f(x, y) = 417.20124599684016
Iteration = 6: x = 9.109591472675179, y = -9.111985672130615, f(x, y) = 406.93259979546116
Iteration = 7: x = 9.000104197548627, y = -9.003135439681682, f(x, y) = 397.1578339610077
Iteration = 8: x = 8.89362822870796, y = -8.89733221762003, f(x, y) = 387.7686597397719
Iteration = 9: x = 8.789422885473126, y = -8.793832957946387, f(x, y) = 378.6909303483104
Iteration = 10: x = 8.68695982889477, y = -8.69210756859238, f(x, y) = 369.87204556222287
Iteration = 11: x = 8.585851986212445, y = -8.591767632247333, f(x, y) = 361.27369158150833
Iteratio

### 9. 1)-4 Final x value and f value

In [5]:
print(f"Final x = {x:.6f}, final y = {y:.6f}")  # final values of the variables after 100 iterations
print(f"Final f(x, y) = {f(x, y):.6f}")  # final value of the objective function
print(f"True minimum: x = -1, y = {-5/6:.6f}, f = {f(-1, -5/6):.6f}")  # analytic minimum, for comparison

Final x = 0.396640, final y = -1.000496
Final f(x, y) = 0.901704
True minimum: x = -1, y = -0.833333, f = -3.083333


### 9. 2) [Bonus] AdaDelta

It keeps the same $G_t$ as RMSProp and adds a second moving average $S_t$ of the squared **updates** $\Delta w_t$:

$S_t = \gamma S_{t-1} + (1-\gamma)(\Delta w_t)^2$

The learning rate $\eta$ is replaced by $\sqrt{S_{t-1} + \epsilon}$, so the step has the same units as $w$:

$w_{t+1} = w_t - \frac{\sqrt{S_{t-1} + \epsilon}}{\sqrt{G_t + \epsilon}}\nabla_w J(w_t)$

Order inside each iteration: compute the gradient, update $G_t$, compute $\Delta w_t$ using the **previous** $S_{t-1}$, update the variable, and only then update $S_t$ with the new $\Delta w_t$.
The same $\gamma$, $\epsilon$, starting point and number of iterations are used, so the result can be compared with RMSProp.

In [6]:
# Implement AdaDelta
gt_x, gt_y = 0, 0  # G_t (moving average of squared gradients) for x and y, reset to 0
st_x, st_y = 0, 0  # S_t (moving average of squared updates) for x and y, start at 0
x, y = 10.0, -10.  # same starting point as RMSProp

for i in range(no_iterations):  
    grad_x = f_der_x(x)  # gradient with respect to x
    grad_y = f_der_y(y)  # gradient with respect to y

    gt_x = gamma * gt_x + (1 - gamma) * grad_x**2  # G_t = γ G_(t-1) + (1-γ) (∂f/∂x)^2
    gt_y = gamma * gt_y + (1 - gamma) * grad_y**2  # G_t = γ G_(t-1) + (1-γ) (∂f/∂y)^2

    delta_x = -np.sqrt(st_x + epsilon) / np.sqrt(gt_x + epsilon) * grad_x  # Δx_t = -√(S_(t-1)+ϵ)/√(G_t+ϵ) · ∂f/∂x
    delta_y = -np.sqrt(st_y + epsilon) / np.sqrt(gt_y + epsilon) * grad_y  # Δy_t = -√(S_(t-1)+ϵ)/√(G_t+ϵ) · ∂f/∂y

    x += delta_x  # x_(t+1) = x_t + Δx_t
    y += delta_y  # y_(t+1) = y_t + Δy_t

    st_x = gamma * st_x + (1 - gamma) * delta_x**2  # S_t = γ S_(t-1) + (1-γ) (Δx_t)^2
    st_y = gamma * st_y + (1 - gamma) * delta_y**2  # S_t = γ S_(t-1) + (1-γ) (Δy_t)^2

    print(f"Iteration = {i + 1}: x = {x}, y = {y}, f(x, y) = {f(x, y)}")  # show the progress

print(f"\nAdaDelta final x = {x:.6f}, final y = {y:.6f}, final f(x, y) = {f(x, y):.6f}")  # final result of AdaDelta

Iteration = 1: x = 9.977639323112486, y = -9.977639322072992, f(x, y) = 488.7887929304578
Iteration = 2: x = 9.954090431199596, y = -9.954094703867618, f(x, y) = 486.46572495115504
Iteration = 3: x = 9.92971872265359, y = -9.929733507926109, f(x, y) = 484.06765778830993
Iteration = 4: x = 9.904700237795812, y = -9.904732982069486, f(x, y) = 481.61251598030594
Iteration = 5: x = 9.879137595900424, y = -9.879196600907388, f(x, y) = 479.1108630946714
Iteration = 6: x = 9.853097501199642, y = -9.853191726680414, f(x, y) = 476.5696537158599
Iteration = 7: x = 9.826626514696926, y = -9.826765454499611, f(x, y) = 473.9938139982198
Iteration = 8: x = 9.799758834195886, y = -9.799952432308539, f(x, y) = 471.3870226185717
Iteration = 9: x = 9.772520571670517, y = -9.772779162164785, f(x, y) = 468.75214078063175
Iteration = 10: x = 9.744932297240947, y = -9.745266559600624, f(x, y) = 466.0914682995907
Iteration = 11: x = 9.71701064483644, y = -9.717431569755187, f(x, y) = 463.4069054129235
Iterat

## 10. Backpropagation

Network structure
- 2 input neurons
- 2 hidden neurons
- 1 output neuron
- Uses sigmoid activation
- Trains on one example using mean squared error

### 10.-1) Sigmoid

In [7]:
import numpy as np  # numerical library for arrays and matrix products

def sigmoid(x):  # sigmoid activation function
    return 1 / (1 + np.exp(-x))  # σ(x) = 1 / (1 + e^(-x))

### 10.-2) Derivative of the sigmoid

$\sigma'(z) = \sigma(z)(1 - \sigma(z))$

In [8]:
def sigmoid_derivative(x):  # derivative of the sigmoid, x is the input z of the sigmoid
    s = sigmoid(x)  # σ(x)
    return s * (1 - s)  # σ'(x) = σ(x)(1 - σ(x))

## Setup

In [9]:
np.random.seed(42)  # fix the random seed so the random initialization is reproducible

# For simplicity, process 1 training example
X = np.array([[0.5, 0.1]])  # input, shape (1, 2)
y = np.array([[1]])  # true output, shape (1, 1)

# Initialize weights and biases with random values
input_size = 2  # number of input neurons
hidden_size = 2  # number of hidden neurons
output_size = 1  # number of output neurons

W1 = np.random.rand(input_size, hidden_size)  # weights input -> hidden, shape (2, 2)
W2 = np.random.rand(hidden_size, output_size)  # weights hidden -> output, shape (2, 1)
b1 = np.random.rand(1, hidden_size)  # biases of the hidden nodes, shape (1, 2)
b2 = np.random.rand(1, output_size)  # bias of the output node, shape (1, 1)

learning_rate = 0.1  # η
epochs = 2000  # num iterations

### 10.-3) Forward computing: final output, class prediction and error

1. Hidden layer: `hidden_input = X·W1 + b1`, `hidden_output = σ(hidden_input)`, shape (1, 2).
2. Output layer: `final_input = hidden_output·W2 + b2`, shape (1, 1), and `final_output = σ(final_input)`.
3. Class prediction: `y_pred = 1` if `final_output ≥ 0.5`, else `0`.
4. Squared error: $E = \frac{1}{2}(y - o)^2$.

In [10]:
def forward(X, y, W1, b1, W2, b2):  
    hidden_input = np.dot(X, W1) + b1  # net of the hidden nodes, shape (1, 2)
    hidden_output = sigmoid(hidden_input)  # output of the hidden nodes, shape (1, 2)
    final_input = np.dot(hidden_output, W2) + b2  # net of the output node, shape (1, 1)
    final_output = sigmoid(final_input)  # output of the output node o, shape (1, 1)
    y_pred = (final_output >= 0.5).astype(int)  # class prediction: 1 if o >= 0.5, else 0
    loss = 0.5 * np.sum((y - final_output) ** 2)  # squared error E = 1/2 (y - o)^2
    return hidden_input, hidden_output, final_input, final_output, y_pred, loss  

### 10.-4) Backward computing, gradients and weight updates

**Output layer** Error term of the output node:

$\delta_{out} = (y - o)\,\sigma'(net_{out})$.
The gradients of $E$ are $dW2 = -\,h^T \delta_{out}$ (shape (2, 1)) and $db2 = -\,\delta_{out}$ (the input of a bias is 1).

**Hidden layer** Each hidden node receives the error of the output node through its weight:

$\delta_{hid} = (\delta_{out} W2^T) \odot \sigma'(net_{hid})$, shape (1, 2).
Then $dW1 = -\,X^T \delta_{hid}$ (shape (2, 2)) and $db1 = -\,\delta_{hid}$.

**Update.** $W \leftarrow W - \eta\, dW$ for every weight and bias. All gradients are computed **before** any update,
so $\delta_{hid}$ uses the old $W2$

In [11]:
def backward(X, y, W1, b1, W2, b2, hidden_input, hidden_output, final_input, final_output, learning_rate):
    delta_out = (y - final_output) * sigmoid_derivative(final_input)  # δ_out = (y - o) σ'(net_out), shape (1, 1)
    dW2 = -np.dot(hidden_output.T, delta_out)  # ∂E/∂W2 = -h^T δ_out, shape (2, 1)
    db2 = -delta_out  # ∂E/∂b2 = -δ_out (bias input is 1), shape (1, 1)

    delta_hidden = np.dot(delta_out, W2.T) * sigmoid_derivative(hidden_input)  # δ_hid = (δ_out W2^T) ⊙ σ'(net_hid), uses the old W2, shape (1, 2)
    dW1 = -np.dot(X.T, delta_hidden)  # ∂E/∂W1 = -X^T δ_hid, shape (2, 2)
    db1 = -delta_hidden  # ∂E/∂b1 = -δ_hid, shape (1, 2)

    W2 = W2 - learning_rate * dW2  # update output weights: W2 = W2 - η dW2
    b2 = b2 - learning_rate * db2  # update output bias: b2 = b2 - η db2
    W1 = W1 - learning_rate * dW1  # update hidden weights: W1 = W1 - η dW1
    b1 = b1 - learning_rate * db1  # update hidden biases: b1 = b1 - η db1
    return W1, b1, W2, b2

### Training loop (backpropagation)

In [12]:
for epoch in range(epochs):  # backpropagation loop over the epochs
    hidden_input, hidden_output, final_input, final_output, y_pred, loss = forward(X, y, W1, b1, W2, b2)  # forward computing
    W1, b1, W2, b2 = backward(X, y, W1, b1, W2, b2, hidden_input, hidden_output, final_input, final_output, learning_rate)  # backward computing and update

    # Print loss value
    if epoch % 50 == 0:  # every 50 epochs
        print(f"Epoch {epoch}, Loss: {loss:.4f}") 

Epoch 0, Loss: 0.0470
Epoch 50, Loss: 0.0229
Epoch 100, Loss: 0.0143
Epoch 150, Loss: 0.0101
Epoch 200, Loss: 0.0077
Epoch 250, Loss: 0.0062
Epoch 300, Loss: 0.0051
Epoch 350, Loss: 0.0044
Epoch 400, Loss: 0.0038
Epoch 450, Loss: 0.0034
Epoch 500, Loss: 0.0030
Epoch 550, Loss: 0.0027
Epoch 600, Loss: 0.0025
Epoch 650, Loss: 0.0023
Epoch 700, Loss: 0.0021
Epoch 750, Loss: 0.0020
Epoch 800, Loss: 0.0018
Epoch 850, Loss: 0.0017
Epoch 900, Loss: 0.0016
Epoch 950, Loss: 0.0015
Epoch 1000, Loss: 0.0014
Epoch 1050, Loss: 0.0014
Epoch 1100, Loss: 0.0013
Epoch 1150, Loss: 0.0012
Epoch 1200, Loss: 0.0012
Epoch 1250, Loss: 0.0011
Epoch 1300, Loss: 0.0011
Epoch 1350, Loss: 0.0010
Epoch 1400, Loss: 0.0010
Epoch 1450, Loss: 0.0010
Epoch 1500, Loss: 0.0009
Epoch 1550, Loss: 0.0009
Epoch 1600, Loss: 0.0009
Epoch 1650, Loss: 0.0008
Epoch 1700, Loss: 0.0008
Epoch 1750, Loss: 0.0008
Epoch 1800, Loss: 0.0008
Epoch 1850, Loss: 0.0007
Epoch 1900, Loss: 0.0007
Epoch 1950, Loss: 0.0007


### Final prediction
Since the target is $y = 1$, the output $o$ should increase toward 1 during training and the loss should decrease toward 0

In [13]:
# Final prediction value
print("\nFinal prediction:", y_pred)  # predicted class (1 or 0)
print("Final output o:", final_output)  # raw output of the network after training


Final prediction: [[1]]
Final output o: [[0.96324573]]
